In [1]:
import pandas as pd
import os
from pathlib import Path

import sys
sys.path.append('../src') # add src directory to path

from utils import format_discussion

In [2]:
# parent directory
parent_dir = Path.cwd().parent

# path to the dataset
data_path = os.path.join(parent_dir ,'data','raw','ParlEE_UK_plenary_speeches.csv')

# reading a CSV file into a DataFrame
df = pd.read_csv(data_path)

C:\Users\FedeF\AppData\Local\Temp\ipykernel_39940\2812760694.py:8: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


In [3]:
df.head()

,instance_id,date,agenda,speechnumber,sentencenumber,speaker,party,text,parliament,iso3country,chair,eu,policyarea,cmp_party
0,1,12/01/2009,Defence,1,1,Simon Hughes,LibDem,What recent assessment he has made of the risk...,UK-HouseOfCommons,GBR,False,0,16,51421.0
1,2,12/01/2009,Defence,2,1,John Hutton,Lab,"Before I begin, I am sure that the whole House...",UK-HouseOfCommons,GBR,False,0,19,51320.0
2,3,12/01/2009,Defence,2,2,John Hutton,Lab,"They were Sergeant Christopher Reed, of 6th Ba...",UK-HouseOfCommons,GBR,False,0,16,51320.0
3,4,12/01/2009,Defence,2,3,John Hutton,Lab,Our thoughts and prayers are likewise with the...,UK-HouseOfCommons,GBR,False,0,19,51320.0
4,5,12/01/2009,Defence,2,4,John Hutton,Lab,Providing effective help and support for servi...,UK-HouseOfCommons,GBR,False,0,16,51320.0


## Formatting discussions


To identify a single debate in our dataset, we decided to consider the features 'agenda' and 'date'. In our dataset, each entry represents a single sentence said in a debate by a politician. Through the function ```format_discussion()``` we're able to join all sentences to reconstruct the debate.

In [4]:
sample_discussion = format_discussion(df, agenda="Defence", date="12/01/2009")
print(sample_discussion)

Simon Hughes (LibDem): What recent assessment he has made of the risk of overstretch affecting forces' families.
John Hutton (Lab): Before I begin, I am sure that the whole House will wish to join me in sending our profound condolences to the families and friends of the servicemen killed in Afghanistan since the House last met. They were Sergeant Christopher Reed, of 6th Battalion the Rifles, and Corporal Robert Deering, Corporal Liam Elms and Lance Corporal Ben Whatley, all of them Royal Marines. Our thoughts and prayers are likewise with the family of the Royal Marine who died in Afghanistan yesterday. Providing effective help and support for service families remains a high priority for the Ministry of Defence. We recognise the challenges that service families face as a result of the current high level of operations and have already made a significant investment-for example, in service accommodation and in making it easier for service personnel and their families to keep in touch dur

Now i need to create a new DataFrame, where for each agenda-date pair, it has a 'discussion' entry that contains the whole discussion. I need to use the 'format_discussion()' method.

Since ParlEE_eng is more than 1.5 GB, this task has to be performed in a very efficient way.

In [ ]:
# TODO: find an efficient way to create the dataset for summarization

# create a smaller dataframe with relevant columns
df_temp = df[['date', 'agenda', 'speechnumber', 'sentencenumber', 'speaker', 'party', 'text']]
df_temp.head()


,date,agenda,speechnumber,sentencenumber,speaker,party,text
0,12/01/2009,Defence,1,1,Simon Hughes,LibDem,What recent assessment he has made of the risk...
1,12/01/2009,Defence,2,1,John Hutton,Lab,"Before I begin, I am sure that the whole House..."
2,12/01/2009,Defence,2,2,John Hutton,Lab,"They were Sergeant Christopher Reed, of 6th Ba..."
3,12/01/2009,Defence,2,3,John Hutton,Lab,Our thoughts and prayers are likewise with the...
4,12/01/2009,Defence,2,4,John Hutton,Lab,Providing effective help and support for servi...


In [14]:
df_temp.groupby(['agenda', 'date']).head()

,date,agenda,speechnumber,sentencenumber,speaker,party,text
0,12/01/2009,Defence,1,1,Simon Hughes,LibDem,What recent assessment he has made of the risk...
1,12/01/2009,Defence,2,1,John Hutton,Lab,"Before I begin, I am sure that the whole House..."
2,12/01/2009,Defence,2,2,John Hutton,Lab,"They were Sergeant Christopher Reed, of 6th Ba..."
3,12/01/2009,Defence,2,3,John Hutton,Lab,Our thoughts and prayers are likewise with the...
4,12/01/2009,Defence,2,4,John Hutton,Lab,Providing effective help and support for servi...
...,...,...,...,...,...,...,...
6766718,19/12/2019,Electoral Practices,148,1,Steve Baker,Con,It is a privilege to lead the first Adjournmen...
6766719,19/12/2019,Electoral Practices,148,2,Steve Baker,Con,I am sure that the Minister will not mind my s...
6766720,19/12/2019,Electoral Practices,148,3,Steve Baker,Con,"I am especially grateful to you, Mr Speaker, f..."
6766721,19/12/2019,Electoral Practices,148,4,Steve Baker,Con,"On 26 September, I intervened in a general deb..."


In [22]:
# create new dataset with formatted discussions
formatted_discussions = []

# only one loop to group by agenda and date
for (agenda, date), group in df_temp.groupby(['agenda', 'date']):
    discussion_text = format_discussion(group, agenda, date)
    formatted_discussions.append({
        'agenda': agenda,
        'date': date,
        'discussion': discussion_text
    })

discussion_df = pd.DataFrame(formatted_discussions)


In [23]:
# save the df to a csv file
output_path = os.path.join(parent_dir ,'data','processed','discussion_dataset.csv')
discussion_df.to_csv(output_path, index=False)

In [ ]:
print(discussion_df[(discussion_df['agenda'] == 'Defence') & (discussion_df['date'] == "12/01/2009")]['discussion'].values[0])

# compare to the original sample discussion


Simon Hughes (LibDem): What recent assessment he has made of the risk of overstretch affecting forces' families.
John Hutton (Lab): Before I begin, I am sure that the whole House will wish to join me in sending our profound condolences to the families and friends of the servicemen killed in Afghanistan since the House last met. They were Sergeant Christopher Reed, of 6th Battalion the Rifles, and Corporal Robert Deering, Corporal Liam Elms and Lance Corporal Ben Whatley, all of them Royal Marines. Our thoughts and prayers are likewise with the family of the Royal Marine who died in Afghanistan yesterday. Providing effective help and support for service families remains a high priority for the Ministry of Defence. We recognise the challenges that service families face as a result of the current high level of operations and have already made a significant investment-for example, in service accommodation and in making it easier for service personnel and their families to keep in touch dur

## Summarization

After having generated the DataFrame with the whole debates, we can proceed to summarize them.

In [10]:
import ollama 

client = ollama.Client()

model = "llama3.2"
prompt = f"Summarize the following UK parliamentary debate discussion in a concise manner:\n\n{sample_discussion}\n\nSummary:"

response = client.chat(model=model, messages=[{"role": "user", "content": prompt}])

print("Ollama Response:")
print(response.message['content'])

Ollama Response:
The UK parliamentary debate discussion focused on the challenges faced by service families due to the high level of operations and overstretch affecting forces. The Secretary of State for Defence (John Hutton) expressed condolences to the families of servicemen killed in Afghanistan and reiterated the government's commitment to providing effective help and support.

Liberal Democrat MP Simon Hughes asked about recent assessments of the risk of overstretch affecting forces' families, given the upcoming change in Iraq and the potential reduction in operational tempo. The Secretary of State acknowledged the opportunity to provide better support and promised to continue looking at measures to help families.

Other MPs raised concerns about housing provision for service personnel when they leave the armed forces, the need for greater clarity regarding accommodation options, and the importance of providing morale-boosting information to families while loved ones are on activ

TODO: use ROUGE score to evaluate summary